# 07 - Top models of a study

Shows, for each k, the best configurations of one study next to the daily and 7-day baselines. It computes nothing new: it reads `metrics/regime_metrics.csv`, written by `scripts/06_regime_metrics.py`, so run the script first.

How the configurations are chosen, for each k, with the days of `choose_period` (val) only:
1. Each configuration (model, z_dim, lam, beta, input) is the mean over its seeds, shown as mean +- std. Each input is measured in its matched view: the daily price for unsmoothed inputs, the backward 7-day mean of the price for smoothed ones.
2. Configurations that switch regime more than `persistence_factor` (2) times the switches per year of the 7-day baseline with the same k are left out: they do not make regimes that last weeks or months.
3. The rest are ordered by the F1 of the expensive days and the first `top` are shown.
4. The same configurations are then shown with the days of `report_period` (test), in the same order. The test days are not used to choose.

The El Nino capture and lift of each episode are shown but not used to order: the capture grows with the size of the most expensive regime. Precision, recall and F1 are those of the expensive days in the most expensive regime (a day is expensive when its real price is above the 66.7, 75 or 80 percentile of the training days for k = 3, 4 and 5).

<d.benavidess@uniandes.edu.co>

In [10]:
import sys
sys.path.append("..")

from pathlib import Path

import pandas as pd

from src.metrics import comparison_table, persistence_factor, ranking

## 0. Configuration

`study` is a study folder name of `experiments/` (for example `weekly`, `dimensions`, `lam_beta_grid`). `top` is how many configurations are shown per k. `choose_period` is the period used to filter and order the configurations (val); `report_period` is shown for the same configurations, in the same order, and is not used to choose (test).

In [11]:
study = "weekly"
top = 3
k_values = [3, 4, 5]
choose_period, report_period = "val", "test"

## 1. Top configurations

In [12]:
rows = pd.read_csv(Path("../metrics/regime_metrics.csv"))
if study not in set(rows["study"]):
    raise ValueError(f"{study} is not in metrics/regime_metrics.csv: {sorted(set(rows['study']))}")

baseline_labels = {"features": "Baseline", "view": "View"}
model_labels = {"rank_c": "Rank", "z_dim": "z_dim", "lam": "lam", "beta": "beta", "features": "Features",
                "view": "View", "seeds": "Seeds"}
keys = ["model", "study", "z_dim", "lam", "beta", "features", "view"]

for k in k_values:
    stats = ranking(rows, k, choose_period)
    limit = stats["switches_limit"].iloc[0]
    candidates = stats[stats["study"] == study]
    passed = candidates.dropna(subset=["rank_c"]).sort_values("rank_c").head(top)
    passed = passed.assign(rank_c=range(1, len(passed) + 1))
    print(f"k = {k}: {study}, {len(candidates)} configurations, {len(candidates.dropna(subset=['rank_c']))} switch at most "
          f"{persistence_factor:g} x the 7-day baseline in {choose_period} ({limit:.1f} per year)")
    for period in (choose_period, report_period):
        period_stats = stats if period == choose_period else ranking(rows, k, period)
        print(f"Period: {period}" + (" (chooses the models)" if period == choose_period else " (reported)"))
        print(comparison_table(period_stats[period_stats["model"] == "level_only"], baseline_labels))
        if passed.empty:
            print(f"No configuration of {study} experiments passed the switches filter with k = {k}.")
        else:
            shown = passed[keys + ["rank_c"]].merge(period_stats.drop(columns=["rank_c"]), on=keys)
            print(comparison_table(shown.sort_values("rank_c"), model_labels))
    print()

k = 3: weekly, 9 configurations, 0 switch at most 2 x the 7-day baseline (30.7 per year)
+----------+-------+---------------+-------------------+-----------------+------------+-----------------+------------+-------------+----------+------+
| Baseline |  View | Switches / yr | Top regime days % | Capture 15-16 % | Lift 15-16 | Capture 23-24 % | Lift 23-24 | Precision % | Recall % | F1 % |
+----------+-------+---------------+-------------------+-----------------+------------+-----------------+------------+-------------+----------+------+
|  level   | daily |      64.7     |        33.6       |       89.2      |    2.65    |       87.2      |    2.60    |     90.6    |   91.4   | 91.0 |
| level 7d |   7d  |      15.4     |        34.1       |       91.1      |    2.67    |       90.4      |    2.65    |     90.8    |   93.1   | 91.9 |
+----------+-------+---------------+-------------------+-----------------+------------+-----------------+------------+-------------+----------+------+
No co